### Pathways detection

In [2]:
import json
import gc
import os
import re
import torch
import concurrent.futures
from pathlib import Path
from collections import defaultdict

from tqdm import tqdm
from vllm import LLM, SamplingParams
from vllm.sampling_params import LogitsProcessor


/DATA/DATANAS2/rhh25/Cellhit/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-08 03:42:06,688	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
library_path = Path('/villa/rhh25/Cellhit/') 
os.chdir(library_path)

In [4]:

from CellHit.LLMs import generate_prompt
from CellHit.data import get_reactome_layers

In [5]:
# ── LLM (vLLM — batched, fast) ────────────────────────────────────────────────
model_path = Path('/villa/rhh25/Cellhit/mixtral/').expanduser()

llm = LLM(
    model=str(model_path),
    revision="gptq-4bit-32g-actorder_True",
    quantization="gptq",
    dtype='float16',
    gpu_memory_utilization=0.95,
    enforce_eager=True,
)




WARNING 04-08 03:42:09 config.py:618] Casting torch.bfloat16 to torch.float16.
WARNING 04-08 03:42:09 config.py:193] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 04-08 03:42:09 llm_engine.py:87] Initializing an LLM engine with config: model='/villa/rhh25/Cellhit/mixtral', tokenizer='/villa/rhh25/Cellhit/mixtral', tokenizer_mode=auto, revision=gptq-4bit-32g-actorder_True, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, disable_custom_all_reduce=False, quantization=gptq, enforce_eager=True, kv_cache_dtype=auto, device_config=cuda, seed=0)
INFO 04-08 03:42:36 llm_engine.py:357] # GPU blocks: 24802, # CPU blocks: 2048


In [8]:


# ── Paths ─────────────────────────────────────────────────────────────────────
data_path          = Path('/DATA/DATANAS2/rhh25/Cellhit/data/')

select_prompt_path = data_path / 'prompts' / 'mixtral_pathway_selector.txt'
drug_folder        = library_path/'drug_descriptions'
output_folder      = library_path / 'drug_pathways'
output_folder.mkdir(exist_ok=True)

# ── Tunable knobs ─────────────────────────────────────────────────────────────
BATCH_SIZE     = 2   # drugs per batched LLM call (tune to VRAM)
PATHWAY_NUMBER = 15   # pathways to select per drug
SELF_K         = 5    # self-consistency repetitions
TEMPERATURE    = 0.7
WRITE_WORKERS  = 8

# ── Reactome pathways ─────────────────────────────────────────────────────────
reactome_pathways = (
    get_reactome_layers(data_path / 'reactome', layer_number=1)['PathwayName']
    .str.strip()
    .tolist()
)
pathway_set = set(reactome_pathways)




# ── Prompt builder ────────────────────────────────────────────────────────────

def build_prompt(drug_description: str) -> str:
    return generate_prompt(
        select_prompt_path,
        drug_description=drug_description,
        pathways_list="\n".join(reactome_pathways),
        pathway_number=PATHWAY_NUMBER,
    )


# ── Output parser ─────────────────────────────────────────────────────────────
# Matches lines like:  Pathway 3: Signal Transduction
_PATHWAY_RE = re.compile(r'Pathway\s+\d+\s*:\s*(.+)', re.IGNORECASE)
_RATIONALE_RE = re.compile(r'Rationale\s+\d+\s*:\s*(.+)', re.IGNORECASE)


def parse_output(text: str) -> dict:
    """
    Extract (rationale, pathway) pairs from free-form LLM output.
    Falls back to fuzzy matching against the known pathway list if the
    exact name isn't found.
    """
    lines      = text.splitlines()
    pathways   = []
    rationales = []

    for line in lines:
        pm = _PATHWAY_RE.match(line.strip())
        if pm:
            name = pm.group(1).strip()
            # exact match first
            if name in pathway_set:
                pathways.append(name)
            else:
                # fuzzy: pick the pathway with the most overlapping words
                name_words = set(name.lower().split())
                best = max(reactome_pathways,
                           key=lambda p: len(set(p.lower().split()) & name_words))
                pathways.append(best)

        rm = _RATIONALE_RE.match(line.strip())
        if rm:
            rationales.append(rm.group(1).strip())

    # pad/trim to PATHWAY_NUMBER
    pathways   = (pathways   + [''] * PATHWAY_NUMBER)[:PATHWAY_NUMBER]
    rationales = (rationales + [''] * PATHWAY_NUMBER)[:PATHWAY_NUMBER]
    return {'pathway_names': pathways, 'pathway_rationales': rationales}


# ── Self-consistency aggregator ───────────────────────────────────────────────

def self_consistency(dict_list: list, normalize: bool = False) -> dict:
    out = {}
    for d in dict_list:
        for pathway, rationale in zip(d['pathway_names'], d['pathway_rationales']):
            if not pathway:
                continue
            if pathway not in out:
                out[pathway] = {'count': 0, 'rationales': []}
            out[pathway]['count'] += 1
            out[pathway]['rationales'].append(rationale)
    if normalize:
        n = len(dict_list)
        for v in out.values():
            v['count'] /= n
    return out


# ── File helpers ──────────────────────────────────────────────────────────────

def already_done(drug_name: str) -> bool:
    return (output_folder / f'{drug_name}_pathways.json').exists()


def _write_one(args):
    drug_name, result = args
    path = output_folder / f'{drug_name}_pathways.json'
    path.write_text(json.dumps(result, indent=2))


def write_batch(drug_names, results):
    with concurrent.futures.ThreadPoolExecutor(max_workers=WRITE_WORKERS) as pool:
        pool.map(_write_one, zip(drug_names, results))


# ── Core batch processor ──────────────────────────────────────────────────────

def process_batch(batch_drugs: list[tuple[str, str]]):
    """
    batch_drugs: list of (drug_name, drug_description)

    Strategy:
      - Run SELF_K rounds of batched inference, each round processing
        all drugs in the batch simultaneously.
      - Each round = 1 call to llm.generate() with len(batch) prompts.
      - Total LLM calls = SELF_K  (not SELF_K × len(batch)).
    """
    drug_names   = [d[0] for d in batch_drugs]
    prompts      = [build_prompt(d[1]) for d in batch_drugs]

    # dict_lists[i] accumulates self_k outputs for drug i
    dict_lists: list[list[dict]] = [[] for _ in range(len(batch_drugs))]

    sampling_params = SamplingParams(
        temperature=TEMPERATURE,
        top_p=0.95,
        max_tokens=2048,   # pathway selection output is verbose
    )

    # SELF_K batched rounds — all drugs in parallel each round
    for _ in range(SELF_K):
        outputs = llm.generate(prompts, sampling_params)
        for i, output in enumerate(outputs):
            text   = output.outputs[0].text
            parsed = parse_output(text)
            dict_lists[i].append(parsed)

    # Aggregate and write
    results = [self_consistency(dl) for dl in dict_lists]
    write_batch(drug_names, results)


# ── Main ──────────────────────────────────────────────────────────────────────

if __name__ == '__main__':

    # Collect all pending drugs
    all_drugs = []
    for file in sorted(drug_folder.glob('*.txt')):
        drug_name = file.stem.replace('_description_refined', '')
        if not already_done(drug_name):
            all_drugs.append((drug_name, file.read_text()))

    print(f"{len(all_drugs)} drugs pending")

    # Split into mini-batches
    batches = [
        all_drugs[i:i + BATCH_SIZE]
        for i in range(0, len(all_drugs), BATCH_SIZE)
    ]

    for batch in tqdm(batches, desc='Batches', unit='batch'):
        process_batch(batch)

    print("All done.")

4 drugs pending


Batches: 100%|██████████| 2/2 [10:53<00:00, 326.50s/batch]

All done.


In [15]:
output_dict = self_consistency(dict_list)
output_dict

{'Apoptosis': {'count': 1, 'rationales': ['']},
 'Cell Cycle Checkpoints': {'count': 1, 'rationales': ['']},
 'MTOR signalling': {'count': 1, 'rationales': ['']},
 None: {'count': 24,
  'rationales': [None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None]},
 'Cell Cycle, Mitotic': {'count': 1,
  'rationales': [" Docetaxel primarily binds to and stabilizes microtubules, promoting microtubule polymerization and inhibiting disassembly. This interference with microtubule dynamics prevents the formation of the mitotic spindle, leading to cell cycle arrest and apoptosis in proliferating cells. The primary target of docetaxel is the beta-tubulin subunit of microtubules, specifically the TUBB protein. Therefore, pathways involved in microtubule dynamics, cell cycle regulation, and apoptosis are likely to be relevant to docetaxel's mechani

In [16]:
#return only pathways with count > 2
for key in output_dict.keys():
    if output_dict[key]['count'] >= 0:
        print(key)

Apoptosis
Cell Cycle Checkpoints
MTOR signalling
None
Cell Cycle, Mitotic
Adaptive Immune System
Degradation of the extracellular matrix


In [20]:
# Iterate and process one by one
for file in folder.glob('*.txt'):
    text = file.read_text()
    print(file.name, len(text))


Camptothecin_description_refined.txt 2125
Cisplatin_description_refined.txt 2772
Cytarabine_description_refined.txt 3705
Vinblastine_description_refined.txt 2714


In [ ]:
@guidance
def inference(lm,prompt, pathway_number, pathways_list,temperature=0.7):

    break_line = "\n"

    lm += prompt

    choosen_pathways = []
    #controlled generation
    for i in range(pathway_number):
        feasible_pathways = [i for i in pathways_list if i not in set(choosen_pathways)]
        lm += f"\nRationale {i+1}:{gen(f'rationale_{i+1}',stop=break_line,temperature=temperature)}"
        lm += f"\nPathway {i+1}:{select(options=feasible_pathways,name=f'pathway_{i+1}')}\n"
        choosen_pathways.append(lm[f'pathway_{i+1}'])
    return lm

def self_consistency(dict_list,normalize=False):

    out = {}

    for d in dict_list:
        
        for pathway,rationale in zip(d['pathway_names'],d['pathway_rationales']):
            if pathway not in out.keys():
                out[pathway] = {}
                out[pathway]['count'] = 1
                out[pathway]['rationales'] = [rationale]
            else:
                out[pathway]['count'] += 1
                out[pathway]['rationales'].append(rationale)

    #devide rationales by count
    if normalize:
        for key in out.keys():
            out[key]['count'] = out[key]['count']/len(dict_list)

    return out


reactome_pathways = get_reactome_layers(data_path/'reactome',layer_number=1)['PathwayName'].tolist()

gpu_id = 0
lm = models.Transformers(str(model_path),**{"device_map":f"cuda:{gpu_id}","revision":"gptq-4bit-32g-actorder_True"})

pathway_number = 15
self_k = 5
temperature = 0.7

# with open('drug_description_refined.txt','r') as f:
#     drug_description = f.read()

drug_folder = library_path/'drug_descriptions'


for file in drug_folder.glob('*.txt'):
    drug_description = file.read_text()
    drug_name = file.name.split('_')[0]

    prompt = generate_prompt(data_path/'prompts'/'mixtral_pathway_selector.txt',**{'drug_description':drug_description,'pathways_list':reactome_pathways,'pathway_number':pathway_number})

    dict_list = []

    for iter in range(self_k):
        lm = lm + inference(prompt,pathway_number=pathway_number, pathways_list=reactome_pathways,temperature=temperature)
        dict_list.append(dictionary_maker(lm))
        lm.reset()
        gc.collect()
        torch.cuda.empty_cache()

    output_dict = self_consistency(dict_list)


    with open(f"{data_path}/{drug_name}_pathways.json", "w") as f:
        json.dump(output_dict, f)

In [29]:
 with open("test_pathway.json", "w") as f:
        json.dump(output_dict, f)

In [3]:
from CellHit.LLMs import fetch_abstracts, generate_prompt
from CellHit.data import obtain_drugs_metadata

In [4]:
#specify the data_path
data_path = library_path/'data'
describe_prompt_path = data_path/'prompts'/'drug_description_prompt.txt'
refine_prompt_path = data_path/'prompts'/'drug_refiner_prompt.txt'
model_path = Path('~/Cellhit/mixtral/').expanduser() # note that the model needs to be downloaded from huggingface, the one used can be found at https://huggingface.co/TheBloke/Mixtral-8x7B-Instruct-v0.1-GPTQ (gptq-4bit-32g-actorder_True)

In [5]:
dataset = 'prism'
metadata = obtain_drugs_metadata(dataset,data_path)

In [6]:
metadata.dropna(subset=['MOA', 'repurposing_target'] )

,Drug,BroadID,DrugID,MOA,repurposing_target
2,"1,4-BUTANEDIOL",BRD:BRD-K22346679-001-06-3,2,"BENZODIAZEPINE RECEPTOR AGONIST, GAMMA HYDROXY...","MAN1B1, PLA2G2A, PLA2G2E"
7,1-AZAKENPAULLONE,BRD:BRD-K03273112-001-01-8,7,GLYCOGEN SYNTHASE KINASE INHIBITOR,"CCNB1, CDK1, CDK5, GSK3B"
8,1-EBIO,BRD:BRD-K70586315-001-02-4,9,POTASSIUM CHANNEL ACTIVATOR,"KCNN1, KCNN2, KCNN3, KCNN4"
12,1-PHENYLBIGUANIDE,BRD:BRD-K31491153-003-05-9,14,SEROTONIN RECEPTOR AGONIST,"HTR3A, HTR3B"
14,10-DEBC,BRD:BRD-K70792160-003-04-6,16,AKT INHIBITOR,PIM1
...,...,...,...,...,...
6426,ZM-336372,BRD:BRD-K73789395-001-10-7,6314,RAF INHIBITOR,"BRAF, LCK, MAPK14, RAF1"
6427,ZOFENOPRIL-CALCIUM,BRD:BRD-K46654563-238-01-0,6317,ANGIOTENSIN CONVERTING ENZYME INHIBITOR,ACE
6428,ZOLPIDEM,BRD:BRD-K44876623-001-03-7,6323,BENZODIAZEPINE RECEPTOR AGONIST,"GABRA1, GABRA2, GABRA3"
6429,ZONIPORIDE,BRD:BRD-K43992824-300-01-0,6326,SODIUM/HYDROGEN EXCHANGER INHIBITOR,SLC9A1


In [ ]:
import gc
import torch
import guidance
import pandas as pd
from guidance import select,gen,models
from pathlib import Path

from CellHit.data import get_reactome_layers,get_pathway_genes
from CellHit.LLMs import generate_prompt,dictionary_maker,self_consistency

NameError: name 'LLM' is not defined

In [18]:
import os
import concurrent.futures
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from vllm import LLM, SamplingParams

from CellHit.LLMs import fetch_abstracts, generate_prompt
from CellHit.data import obtain_drugs_metadata

# ── Paths ─────────────────────────────────────────────────────���───────────────
library_path         = Path('/villa/rhh25/Cellhit/')
data_path            = library_path / 'data'
describe_prompt_path = data_path / 'prompts' / 'drug_description_prompt.txt'
refine_prompt_path   = data_path / 'prompts' / 'drug_refiner_prompt.txt'
model_path           = Path('~/Cellhit/mixtral/').expanduser()

OUTPUT_DIR   = Path('drug_descriptions')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Tunable knobs ─────────────────────────────────────────────────────────────
BATCH_SIZE       = 2   # number of drugs per mini-batch (tune to VRAM / RAM)
ABSTRACT_WORKERS = 2    # parallel threads for PubMed HTTP fetches
WRITE_WORKERS    = 2   # parallel threads for file writes
ABSTRACTS_K      = 10   # number of PubMed abstracts per drug

# ── Dataset / metadata ───────────────────────────────────────────────���────────
dataset          = 'gdsc'
metadata         = obtain_drugs_metadata(dataset, data_path)
metadata_indexed = metadata.set_index('Drug')   # O(1) look-ups
drug_list        = metadata_indexed.index.tolist()[:4]



# ── Helpers ───────────────────────────────────────────────────────────────────

def _fetch_one(drug_name, mail=None, k=10):
    """Fetch PubMed abstracts for a single drug. Returns (name, list[str])."""
    try:
        abstracts = fetch_abstracts(drug_name, mail=mail, k=k)
        return drug_name, [str(a) for a in abstracts]
    except Exception as exc:
        print(f"[WARN] fetch_abstracts failed for '{drug_name}': {exc}")
        return drug_name, []


def fetch_batch_abstracts(batch, mail=None, k=10):
    """Fetch abstracts for every drug in *batch* concurrently."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=ABSTRACT_WORKERS) as pool:
        results = pool.map(lambda name: _fetch_one(name, mail, k), batch)
    return dict(results)


def build_describe_prompt(drug_name, metadata_indexed, describe_prompt_path):
    row    = metadata_indexed.loc[drug_name]
    moa    = row['MOA']                if isinstance(row['MOA'], str)                else "Not annotated"
    target = row['repurposing_target'] if isinstance(row['repurposing_target'], str) else "Not annotated"
    return generate_prompt(describe_prompt_path,
                           drug_name=drug_name,
                           mechanism_of_action=moa,
                           putative_targets=target)


def build_refine_prompt(description, abstracts, refine_prompt_path):
    formatted = "\n".join(f"Abstract {i+1}: {a}" for i, a in enumerate(abstracts))
    return generate_prompt(refine_prompt_path,
                           previous_output=description,
                           formatted_abstracts=formatted)


def _write_one(drug_name, text):
    (OUTPUT_DIR / f'{drug_name}_description_refined.txt').write_text(text)


def write_batch(drug_names, refined_texts):
    """Write output files for a batch concurrently."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=WRITE_WORKERS) as pool:
        pool.map(_write_one, drug_names, refined_texts)


def already_done(drug_name):
    """Skip drugs whose output file already exists (safe resumption)."""
    return (OUTPUT_DIR / f'{drug_name}_description_refined.txt').exists()


# ── Batched pipeline ──────────────────────────────────────────────────────────

def process_batch(batch):
    """
    For a list of drug names:
      1. Fetch abstracts in parallel (I/O).
      2. One batched LLM call  →  descriptions.
      3. One batched LLM call  →  refined descriptions.
      4. Write output files in parallel (I/O).
    """
    # --- 1. Parallel abstract fetch ------------------------------------------
    abstracts_map = fetch_batch_abstracts(batch, mail=None, k=ABSTRACTS_K)

    # --- 2. Build describe prompts & single batched inference ----------------
    describe_prompts = [
        build_describe_prompt(name, metadata_indexed, describe_prompt_path)
        for name in batch
    ]
    describe_outputs = llm.generate(describe_prompts, sampling_params)
    descriptions     = [o.outputs[0].text for o in describe_outputs]

    # --- 3. Build refine prompts & single batched inference ------------------
    refine_prompts = [
        build_refine_prompt(desc, abstracts_map.get(name, []), refine_prompt_path)
        for name, desc in zip(batch, descriptions)
    ]
    refine_outputs = llm.generate(refine_prompts, sampling_params)
    refined_texts  = [o.outputs[0].text for o in refine_outputs]

    # --- 4. Parallel file writes ---------------------------------------------
    write_batch(batch, refined_texts)


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == '__main__':

    # Filter out already-completed drugs so the run is safely resumable
    pending = [d for d in drug_list if not already_done(d)]
    print(f"{len(drug_list)} drugs total | {len(pending)} pending | "
          f"{len(drug_list) - len(pending)} already done")

    # Split into mini-batches
    batches = [pending[i:i + BATCH_SIZE] for i in range(0, len(pending), BATCH_SIZE)]

    for batch in tqdm(batches, desc="Batches", unit="batch"):
        process_batch(batch)

    print("All done.")

4 drugs total | 4 pending | 0 already done


Batches:   0%|          | 0/2 [00:00<?, ?batch/s]/villa/rhh25/Cellhit/venv/lib/python3.10/site-packages/Bio/Entrez/__init__.py:694: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


Abstract 1 is not available.

Abstract 2 is not available.

Abstract 3 is not available.

Abstract 4 is not available.

Abstract 7 is not available.



Batches:  50%|█████     | 1/2 [00:59<00:59, 59.57s/batch]

Abstract 1 is not available.

Abstract 4 is not available.

Abstract 6 is not available.



Batches: 100%|██████████| 2/2 [02:12<00:00, 66.06s/batch]

All done.


In [14]:
dataset = 'gdsc'
metadata = obtain_drugs_metadata(dataset,data_path)

drug_name = 'Docetaxel'
def get_refined_drug_description(drug_name, metadata, llm, describe_prompt_path, refine_prompt_path, sampling_params):
    mechanism_of_action = metadata.loc[metadata['Drug']==drug_name]['MOA'].values[0]
    putative_target = metadata.loc[metadata['Drug']==drug_name]['repurposing_target'].values[0]

    #fetch abstracts
    abstracts = fetch_abstracts(drug_name,mail=None,k=10)
    abstracts = [str(a) for a in abstracts]


    describe_prompt = generate_prompt(describe_prompt_path,**{'drug_name':drug_name, 'mechanism_of_action':mechanism_of_action, 'putative_targets':putative_target})

    outputs = llm.generate(describe_prompt, sampling_params)
    drug_description = outputs[0].outputs[0].text

    refine_prompt = generate_prompt(refine_prompt_path,**{'previous_output':drug_description,'formatted_abstracts':"\n".join([f"Abstract {i+1}: {abstract}" for i, abstract in enumerate(abstracts)])})

    outputs = llm.generate(refine_prompt, sampling_params)
    refined_drug_description = outputs[0].outputs[0].text
    #dump refined_drug_description to a file
    with open(f'{drug_name}_description_refined.txt','w') as f:
        f.write(refined_drug_description)